# Merged-Tree Bimodality Diagnostic Pipeline

Checks every cluster in a trial for the "colour says two trees, XY density says one" pattern — two overlapping crowns whose combined footprint never trips `max_radius`, so the production `split_large_clusters()` never even attempts to split them.

**Before running:** update the `sys.path` entries in the next cell to wherever you actually saved `diagnose_cluster_bimodality.py` and `split_large_clusters.py` (the colour-augmented version).

In [1]:
# Initialise the environment, then set the kernel and restart
!bash shells/jupyter_kernel_env.sh

Traceback (most recent call last):
  File "/home/hcr64/.conda/envs/open3d_env/lib/python3.10/runpy.py", line 187, in _run_module_as_main
    mod_name, mod_spec, code = _get_module_details(mod_name, _Error)
  File "/home/hcr64/.conda/envs/open3d_env/lib/python3.10/runpy.py", line 146, in _get_module_details
    return _get_module_details(pkg_main_name, error)
  File "/home/hcr64/.conda/envs/open3d_env/lib/python3.10/runpy.py", line 110, in _get_module_details
    __import__(pkg_name)
  File "/home/hcr64/.conda/envs/open3d_env/lib/python3.10/site-packages/ipykernel/__init__.py", line 7, in <module>
    from .connect import *  # noqa: F403
  File "/home/hcr64/.conda/envs/open3d_env/lib/python3.10/site-packages/ipykernel/connect.py", line 12, in <module>
    import jupyter_client
  File "/home/hcr64/.conda/envs/open3d_env/lib/python3.10/site-packages/jupyter_client/__init__.py", line 4, in <module>
    from .asynchronous import AsyncKernelClient
  File "/home/hcr64/.conda/envs/open3d_env/l

In [2]:
import sys
!{sys.executable} -m pip install plotly open3d
# ── path setup ──────────────────────────────────────────────────────────────
# Adds specific directories so their modules import directly (bare name)
# WITHOUT triggering that directory's __init__.py and its full dependency
# chain (torch/xgboost/lightgbm under modelling/, rasterio/scikit-image/
# pyproj under clustering/). Never `import functions` itself — that name
# collides between clustering/functions and modelling/functions.

sys.path.insert(0, "/home/hcr64/Pinyon-Detection")                                  # global_files/
sys.path.insert(0, "/home/hcr64/Pinyon-Detection/modelling/functions")              # feature_config.py, standalone

# UPDATE THESE TWO to wherever you actually saved the files:
sys.path.insert(0, "/home/hcr64/Pinyon-Detection/modelling/functions/features")     # diagnose_cluster_bimodality.py
sys.path.insert(0, "/home/hcr64/Pinyon-Detection/clustering/functions/detection")   # split_large_clusters.py (colour-augmented)

Defaulting to user installation because normal site-packages is not writeable
ERROR: Could not find a version that satisfies the requirement open3d (from versions: none)
ERROR: No matching distribution found for open3d


In [3]:
# ── third-party ────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt
import plotly.graph_objects as go
# import open3d as o3d

# ── project: paths + cached dataframes/clusters ────────────────────────────
from global_files import load_clusters, load_dataframes, get_paths

# ── project: feature list (standalone import, no torch/xgboost/lightgbm) ───
from feature_config import FEATURES

# ── project: bimodality diagnostics ─────────────────────────────────────────
from diagnose_cluster_bimodality import (
    diagnose_all_clusters,
    diagnose_cluster_bimodality,
    calibrate_thresholds,
    explain_why_not_flagged,
    get_color_spatial_override_candidates,
    get_flagged_cluster_paths,
    get_flagged_indices_from_sources,
)

# ── project: colour-augmented splitting ─────────────────────────────────────
from split_large_clusters import split_large_clusters, find_density_peaks, filter_cluster

print("Imports OK")

ModuleNotFoundError: No module named 'open3d'

In [ ]:
# train the model, make sure modelling is up to date with clusters
!{sys.executable} -u /home/hcr64/Pinyon-Detection/modelling/train_model.py --trial_name Sunset_sfm_trial

## Visualize a flagged cluster (interactive 3D, coloured by RGB)

Uses the point clouds already loaded in memory (`clusters`), not a re-read from disk — faster, and avoids any path-mismatch issues. Subsamples large clusters for responsiveness only; doesn't affect any of the analysis above.

In [ ]:
def plot_pointcloud(pcd, title=None, max_points=20000, point_size=2):
    """
    Interactive 3D scatter of a single Open3D point cloud, coloured by its
    own RGB — for eyeballing whether a flagged cluster's colour-bimodal
    regions actually look like two distinct, spatially separated crowns.
    """
    points = np.asarray(pcd.points)
    colors = np.asarray(pcd.colors) if pcd.has_colors() else None

    if len(points) > max_points:
        idx = np.random.default_rng(0).choice(len(points), max_points, replace=False)
        points = points[idx]
        if colors is not None:
            colors = colors[idx]

    marker_colors = (
        [f"rgb({int(r*255)},{int(g*255)},{int(b*255)})" for r, g, b in colors]
        if colors is not None else "grey"
    )

    fig = go.Figure(data=[go.Scatter3d(
        x=points[:, 0], y=points[:, 1], z=points[:, 2],
        mode="markers",
        marker=dict(size=point_size, color=marker_colors),
    )])
    fig.update_layout(
        title=title or f"n_points={len(points)}",
        scene=dict(aspectmode="data"),
        height=600,
    )
    fig.show()


def plot_cluster(cluster_idx, clusters, **kwargs):
    """Thin wrapper: plot_pointcloud() for a cluster by index into `clusters`."""
    plot_pointcloud(clusters[cluster_idx],
                     title=kwargs.pop("title", f"cluster {cluster_idx}"), **kwargs)

## Load data

Loads the cached dataframes and the actual `.ply` clusters for the trial, and checks they're in sync — `clusters[i]` must correspond to `df_clusters.iloc[i]`, which is only guaranteed if both were produced by the same, uninterrupted `run_clustering.py` run (see prior debugging: `save_clusters()` clears and rewrites the clusters folder on every run, so an older dataframe paired with a newer `.ply` folder — or vice versa — will silently misalign).

In [ ]:
TRIAL_NAME = "Sunset_sfm_trial"
MAX_RADIUS = 4.25   # from pinyons.sh — keep in sync if that value changes

PATHS = get_paths(TRIAL_NAME)

df_clusters, df_deep_clusters = load_dataframes(PATHS['Dataframes'])
clusters = load_clusters(PATHS['Clusters'])

print(f"clusters: {len(clusters)}   df_clusters: {len(df_clusters)}")
assert len(clusters) == len(df_clusters), (
    f"clusters ({len(clusters)}) and df_clusters ({len(df_clusters)}) are out of "
    f"sync — reload both from a single, uninterrupted run before continuing."
)

## (Optional) Calibrate against known problem clusters

If you have a list of clusters already known/suspected to be merged trees (e.g. from `inspect_misclassified_clusters()`), run the raw-stats calibration and the gate-by-gate explanation against them first — this is how the thresholds pinned in the next section were actually chosen, rather than guessed.

**Cluster indices are only valid for the run that produced the `clusters` list above** — they are not stable identifiers across separate `run_clustering.py` runs. Re-identify known clusters by spatial position if you've rerun clustering since these indices were recorded.

In [ ]:
# get the cluster ids of mismatched clusters automatically
df_mismatch = pd.read_csv(PATHS['Dataframes'] + 'pinyon_misclassification_report.csv')

# pull mismatched ids
id_col = "file" if "file" in df_mismatch.columns else df_mismatch.columns[0]
known_problem_indices = df_mismatch[id_col].astype(int).tolist()

# check the mismathces
print(f"{len(known_problem_indices)} mismatched clusters")
print(known_problem_indices)

# calibrate thesholds with the known mismahtces
df_cal = calibrate_thresholds(clusters, known_problem_indices)

# initialise ind fr below
ind = 0

In [ ]:
# look at mismatched clusters, compare with results below
# get the path of desired pointcloud
ind += 1
ind = ind % len( known_problem_indices )
cluster_id = known_problem_indices[ind]

# call the function on it
print( cluster_id )
plot_cluster( cluster_id, clusters )

In [ ]:
explain_why_not_flagged(
    clusters,
    known_problem_indices,
    max_radius=MAX_RADIUS,
)

## Run the full diagnostic

Thresholds below are pinned explicitly (not left as function defaults) so this notebook keeps working even if the defaults in `diagnose_cluster_bimodality.py` change later. Update these values if your own calibration above suggested different cutoffs.

Flags clusters where a colour feature is both statistically bimodal (GaussianMixture, BIC-compared) AND its two colour-assigned point groups occupy spatially distinct sub-regions of the cluster's footprint — the combination that screens out ordinary within-crown colour variation (sun/shade, exterior/interior foliage) while catching genuine spatial merges.

In [ ]:
# thresholds calibrated against known_problem_indices above
DELTA_BIC_THRESHOLD = 10.0
SEPARATION_THRESHOLD = 1.0
SPATIAL_SEPARATION_RATIO = 0.75
MIN_INTERCENTROID_M = 0.3
MIN_SPATIAL_GROUP_SIZE = 15

df_diag = diagnose_all_clusters(
    clusters,
    delta_bic_threshold=DELTA_BIC_THRESHOLD,
    separation_threshold=SEPARATION_THRESHOLD,
    spatial_separation_ratio=SPATIAL_SEPARATION_RATIO,
    min_intercentroid_m=MIN_INTERCENTROID_M,
    min_spatial_group_size=MIN_SPATIAL_GROUP_SIZE,
    save_path=PATHS['Images'] + 'bimodality_diagnostics/',
)

flagged_indices = sorted(df_diag.loc[df_diag["color_bimodal_spatial_unimodal"], "file"].unique().tolist())
print(f"\n{len(flagged_indices)} clusters flagged")

## Sanity check against known cases

Before trusting the full-dataset result, confirm it's consistent with what `explain_why_not_flagged()` already told you above — known confirmed cases should all show up flagged, known rejected cases should not.

In [ ]:
# adjust these to match whatever explain_why_not_flagged printed above
confirmed = {1399, 1375, 1483}   # passed both bimodal + spatial-separation gates
rejected  = {545, 1006, 1744}    # failed the spatial gate or found no bimodality

print("known confirmed present:", confirmed & set(flagged_indices), "  (expect: all of", confirmed, ")")
print("known rejected present: ", rejected & set(flagged_indices), "  (expect: empty set)")

## Split flagged clusters by whether they'd reach the production splitter

Clusters at or below `max_radius` never reach `split_large_clusters()`'s density-peak check at all — gate 1 skips them outright, regardless of how obviously merged they are by colour or point density. Two heavily overlapping crowns can easily produce a compact combined footprint rather than a wide one, so this group is the direct target for the `force_check_indices` override tested below.

In [ ]:
below_gate, above_gate = [], []
for idx in flagged_indices:
    pcd = clusters[idx]
    aabb = pcd.get_axis_aligned_bounding_box()
    radius = max((aabb.max_bound - aabb.min_bound)[:2]) / 2
    (below_gate if radius <= MAX_RADIUS else above_gate).append(idx)

print(f"{len(below_gate)} flagged clusters sit at/below max_radius={MAX_RADIUS} — "
      f"production split_large_clusters() skips these entirely (gate 1)")
print(f"{len(above_gate)} flagged clusters sit above max_radius — "
      f"production would already attempt a split; if still merged, the failure "
      f"is downstream (Mean Shift not converging to 2 modes, or filter_cluster "
      f"rejecting the sub-clusters), not the radius gate")

## Pull a sample of flagged clusters for visual inspection

147+ flagged clusters is too many to eyeball individually — pull a random sample for a first visual pass. `get_flagged_cluster_paths()` also resolves the full flagged set to `.ply` paths on disk, matching `save_clusters()`'s `cluster{N}.ply` naming, for use in other tools.

In [ ]:
paths = get_flagged_cluster_paths(df_diag, PATHS['Clusters'])

random.seed(1)
sample_indices = random.sample(flagged_indices, min(15, len(flagged_indices)))
print("sampled cluster indices:", sample_indices)

In [ ]:
# plot one at a time — re-run with a different index, or uncomment the loop
# below to page through the whole sample
plot_cluster(sample_indices[0], clusters)

# for idx in sample_indices:
#     plot_cluster(idx, clusters)

## Test colour-augmented splitting on one radius-gated candidate

Picks the first `below_gate` cluster (flagged by colour, but never reachable by production's radius gate) and runs the colour-augmented `split_large_clusters()` on just that one cluster, forcing it past the radius gate via `force_check_indices`. `color_weight` is untuned — sweep it against a few known cases before trusting this in production; see the module docstring in `split_large_clusters.py` for the tuning rationale.

In [ ]:
COLOR_WEIGHT = 10.0   # untuned starting point — sweep this

test_idx = below_gate[0] if below_gate else None

if test_idx is not None:
    single_result = split_large_clusters(
        [clusters[test_idx]],
        min_points=40,
        max_radius=MAX_RADIUS,
        min_peak_distance=3.0,
        min_density_ratio=1.5,
        k=40,
        use_color=True,
        color_weight=COLOR_WEIGHT,
        force_check_indices={0},   # position 0 within this single-item list
    )
    print(f"cluster {test_idx}: 1 input -> {len(single_result)} output cluster(s)")
else:
    print("No radius-gated flagged clusters to test — check below_gate above.")

In [ ]:
if test_idx is not None:
    plot_pointcloud(clusters[test_idx], title=f"BEFORE — cluster {test_idx}")
    for i, sub in enumerate(single_result):
        n_pts = len(np.asarray(sub.points))
        plot_pointcloud(sub, title=f"AFTER — piece {i} (n_points={n_pts})")

## (Once the single-cluster test looks right) Batch-test on more candidates

Widen to a larger slice of `below_gate` once the single-cluster result above looks like a genuine two-tree separation. `save_pre_split_path` writes every cluster that reached the Mean-Shift stage (pre-split) to disk with descriptive filenames, for bulk visual QC outside this notebook — same pattern as the existing `pre_split_clusters/` diagnostic folder.

In [ ]:
BATCH_SIZE = 10   # widen once you trust the single-cluster result above

test_indices = below_gate[:BATCH_SIZE]
test_clusters = [clusters[i] for i in test_indices]

batch_result = split_large_clusters(
    test_clusters,
    min_points=40,
    max_radius=MAX_RADIUS,
    min_peak_distance=3.0,
    min_density_ratio=1.5,
    k=40,
    use_color=True,
    color_weight=COLOR_WEIGHT,
    force_check_indices=set(range(len(test_clusters))),   # positions WITHIN this sublist
    save_pre_split_path="/scratch/hcr64/test_override_split/",
)

print(f"{len(test_indices)} input clusters -> {len(batch_result)} output clusters "
      f"after colour-augmented splitting")

## Next steps

- If the batch test above produces sane sub-crowns for most `below_gate` candidates, generate `force_check_indices` from the FULL flagged set (not just this small batch) via `get_color_spatial_override_candidates()` and wire it into a real `run_clustering.py` run.
- Re-sweep `matching_score` after any change to the production pipeline — a change here could shift results for clusters far outside this flagged set too.
- `color_weight` is the main untuned parameter in this whole pipeline — treat any production change as provisional until it's been swept the same way `min_points` / `search_radius_m` were.